# NUMPY MASTERY — CHAPTERS 53–56

```
NUMPY MASTERY
Current Chapters: 53 (Performance Interview), 54 (NumPy vs Pandas),
                  55 (NumPy vs PyTorch/TensorFlow), 56 (Professional Code Quality)
Progress:         56 / 75 chapters
Completed:        Ch1-52
Remaining:        19 chapters
```

Four chapters on judgment and positioning: optimizing code under interview pressure (53),
knowing when to reach for NumPy vs pandas (54) or PyTorch/TensorFlow (55), and writing NumPy
code a senior engineer would approve in code review (56).


In [ ]:
import numpy as np
import timeit
print("NumPy version:", np.__version__)


---
# CHAPTER 53 — NumPy Performance Interview

## The scenario every performance interview question follows
You're handed slow code and asked to optimize it — progressively, with follow-ups probing
*why* each optimization helps, not just *that* it helps. Below is a realistic progression from
naive to production-quality, timed at each stage, matching how a real interview conversation
escalates.

## The starting point
```python
result = []
for i in range(len(x)):
    result.append(x[i] * 2 + 5)
```


In [ ]:
rng = np.random.default_rng(0)
n = 2_000_000
x_list = list(rng.normal(size=n))
x_arr = np.array(x_list)

def v0_python_loop():
    result = []
    for i in range(len(x_list)):
        result.append(x_list[i] * 2 + 5)
    return result

def v1_list_comprehension():
    return [v * 2 + 5 for v in x_list]

def v2_naive_numpy():
    return x_arr * 2 + 5

def v3_numpy_with_out(preallocated):
    np.multiply(x_arr, 2, out=preallocated)
    np.add(preallocated, 5, out=preallocated)
    return preallocated

preallocated = np.empty_like(x_arr)

results = {}
results['v0_python_loop'] = timeit.timeit(v0_python_loop, number=1)
results['v1_list_comprehension'] = timeit.timeit(v1_list_comprehension, number=1)
results['v2_naive_numpy'] = timeit.timeit(v2_naive_numpy, number=1)
results['v3_numpy_with_out'] = timeit.timeit(lambda: v3_numpy_with_out(preallocated), number=1)

baseline = results['v0_python_loop']
for name, t in results.items():
    print(f"{name:22s}: {t:.4f}s   ({baseline/t:.1f}x faster than v0)")


### Expected Output (verified on NumPy 2.4.4 — single-run timings, illustrative of direction/magnitude)
```
v0_python_loop        : 1.1276s   (1.0x faster than v0)
v1_list_comprehension  : 0.3132s   (3.6x faster than v0)
v2_naive_numpy         : 0.0074s   (151.9x faster than v0)
v3_numpy_with_out      : 0.1043s   (10.8x faster than v0)
```

### The follow-up questions an interviewer would actually ask at each step
**"Why is v1 only ~1.6x faster, when I expected NumPy-level gains?"** — v1 is still pure
Python: it removes the explicit `.append()` call overhead and some interpreter bytecode, but
every element still gets boxed/unboxed and dispatched individually (Chapter 1). The `2x` type
of speedup from `for`-loop → list-comprehension is real but categorically smaller than the
`~25x+` jump to true vectorization.

**"Why does v2 (naive numpy) win so much more?"** — `x_arr * 2 + 5` dispatches the entire
operation to compiled C loops once, walking contiguous memory with no per-element Python
overhead (Chapter 1, 12) — this is the actual vectorization gain, not the loop-restructuring
gain from v1.

**"What does v3's `out=` buy you, and is it worth the added complexity?"** — In principle,
avoiding allocation of new output arrays for each intermediate step (Chapter 12's `out=`
parameter) saves some memory-allocation overhead. **But look at the actual measurement above:
v3 came out slower than v2 in this single run** — a live, unplanned demonstration of Chapter
30's core lesson that a single `timeit` call is noisy and can mislead (system scheduling, cache
state, and measurement overhead all vary run to run). A candidate who *trusts* a single
measurement here would wrongly conclude `out=` made things worse; a candidate applying Chapter
30's best-of-N discipline would rerun this with `timeit.repeat` and take the minimum before
drawing any conclusion at all. **This is the answer that separates good from great**: knowing
that `out=`-style micro-optimizations are real but usually secondary to the vectorization jump,
and — just as importantly — refusing to trust a single benchmark number either way, positive or
negative. Reflexively micro-optimizing every line (or reflexively trusting one timing run)
both signal weaker judgment than measuring properly and knowing when a result needs
re-verification before being acted on.

### Interview Insight
> "You're given code with a `for` loop over a NumPy array. Walk me through your optimization
> process."
State the process, not just the answer: (1) confirm with a benchmark that it's actually slow at
the relevant scale — Chapter 29's small-array caveat means don't optimize blindly; (2)
vectorize the core operation first — this is usually the 10-100x win; (3) only then consider
memory-level optimizations (`out=`, dtype, contiguity) if profiling shows they matter; (4)
verify correctness didn't regress (`np.allclose` against the original, Chapter 27) after each
change.


---
# CHAPTER 54 — NumPy vs Pandas

## Concept
Pandas `DataFrame`/`Series` are built **on top of** NumPy arrays — a `DataFrame` column is,
underneath, a NumPy array plus a row-label index and dtype-per-column bookkeeping. Knowing this
relationship tells you exactly when to reach for which tool.

## When to use which
| | NumPy | Pandas |
|---|---|---|
| Homogeneous numerical data (one dtype) | ✅ natural fit | ✅ works, but adds overhead |
| Mixed-type tabular data (strings + numbers + dates) | ❌ awkward (needs structured arrays) | ✅ natural fit |
| Row/column labels matter (named access) | ❌ no built-in labels | ✅ index and column names |
| Missing data handling | manual (`NaN`, masking) | ✅ built-in (`NaN`, `isna()`, `dropna()`) |
| Heavy numerical computation (matrix ops, linear algebra) | ✅ faster, more direct | slower (extra overhead per operation) |
| Grouping/aggregation by category (`groupby`) | manual (`np.unique` + loop, or `bincount`-style) | ✅ built-in, optimized |
| Feeding a model (scikit-learn, PyTorch) | ✅ direct | needs `.values`/`.to_numpy()` conversion first |


In [ ]:
# Demonstrating the actual relationship: a pandas Series IS backed by a numpy array
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

if HAS_PANDAS:
    s = pd.Series([1.0, 2.0, 3.0, np.nan, 5.0])
    print("Series:\n", s)
    print("\ntype(s.values)      :", type(s.values))
    print("s.values is a numpy array:", isinstance(s.values, np.ndarray))
    print("s.values             :", s.values)

    # Operations you already know from Chapter 15 work identically underneath
    print("\nnp.isnan(s.values)   :", np.isnan(s.values))
    print("s.mean() [pandas skips NaN by default, like Chapter 15's np.nanmean]:", s.mean())
    print("np.nanmean(s.values) [the NumPy equivalent]                        :", np.nanmean(s.values))
else:
    print("pandas not installed in this environment -- the conceptual relationship still holds:")
    print("a pandas Series/DataFrame column is fundamentally a NumPy array plus an index/dtype layer.")


### Expected Output (verified on NumPy 2.4.4; pandas availability depends on environment)
```
Series:
 0    1.0
1    2.0
2    3.0
3    NaN
4    5.0
dtype: float64

type(s.values)      : <class 'numpy.ndarray'>
s.values is a numpy array: True
s.values             : [ 1.  2.  3. nan  5.]

np.isnan(s.values)   : [False False False  True False]
s.mean() [pandas skips NaN by default, like Chapter 15's np.nanmean]: 2.75
np.nanmean(s.values) [the NumPy equivalent]                        : 2.75
```

### The result you should expect either way
Whether pandas is installed in your current environment or not, the fact that `s.mean()` and
`np.nanmean(s.values)` agree exactly is the whole point: **pandas' statistical methods are, for
numerical columns, thin wrappers around the exact NumPy functions this entire course has
covered** — `DataFrame.mean()` calls into the same NaN-aware reduction logic as
`np.nanmean` underneath.

### The standard professional workflow: Pandas → NumPy → scikit-learn/PyTorch
```python
df = pd.read_csv("data.csv")              # pandas: heterogeneous loading, cleaning, exploration
df = df.dropna()                            # pandas: convenient missing-data handling
X = df[feature_columns].to_numpy(dtype=np.float32)   # DROP DOWN to NumPy for the numerical core
y = df[target_column].to_numpy()
model.fit(X, y)                             # scikit-learn/PyTorch: expects NumPy arrays directly
```
This is precisely the "Pandas → NumPy → Scikit-learn → Model" chain referenced back in Chapter
1's introduction — pandas owns the messy, heterogeneous data-wrangling phase; NumPy takes over
the moment the data becomes a clean numerical matrix.

### Interview Insight
> "When would you choose NumPy over pandas for a data processing task?"
When the data is already homogeneous numerical data and doesn't need row/column labels, mixed
dtypes, or `groupby`-style categorical aggregation — pure NumPy avoids pandas' per-operation
overhead (index alignment checks, dtype boxing for object columns) and gives more direct control
for heavy numerical/linear-algebra work. In practice, most real pipelines use both: pandas for
loading/cleaning/exploring, then `.to_numpy()` to drop into NumPy for the actual numerical
computation or model training.


---
# CHAPTER 55 — NumPy vs PyTorch / TensorFlow

## Concept
A PyTorch `Tensor` or TensorFlow `Tensor` is, conceptually, "an `ndarray` plus two additional
capabilities NumPy doesn't have": **GPU device placement** and **automatic differentiation**
(autograd) — the mechanism that computes Chapter 46's backpropagation gradients automatically,
rather than by hand.

## Why NumPy fluency directly accelerates learning these frameworks
Every mental model this course built — shape reasoning (Chapter 3, 10), broadcasting (Chapter
11), the `(n,k)@(k,m)->(n,m)` matmul rule (Chapter 22), axis-based reductions (Chapter 10) —
transfers **directly and unchanged** to `torch.Tensor`/`tf.Tensor`. The APIs are close enough
that most NumPy code translates almost line-for-line.


In [ ]:
# Demonstrating the conceptual parity -- NumPy code next to what the PyTorch equivalent looks like
# (shown as commented pseudocode since torch may not be installed in this environment;
#  the POINT is the near-identical API, not requiring torch to be present)

X = np.random.default_rng(0).normal(size=(4, 3)).astype(np.float32)
W = np.random.default_rng(1).normal(size=(3, 2)).astype(np.float32)

numpy_result = X @ W                          # NumPy: Chapter 22's matmul rule
numpy_mean = X.mean(axis=0)                    # NumPy: Chapter 10's axis reduction
numpy_reshaped = X.reshape(2, 6)               # NumPy: Chapter 9's reshape

print("NumPy forward pass  :", numpy_result.shape)
print("NumPy mean(axis=0)  :", numpy_mean.shape)
print("NumPy reshape(2, 6) :", numpy_reshaped.shape)
print()
print("--- PyTorch equivalents (same shapes, same axis semantics, near-identical syntax) ---")
print("import torch")
print("X_t = torch.from_numpy(X)                 # zero-copy conversion FROM a numpy array")
print("W_t = torch.from_numpy(W)")
print("torch_result = X_t @ W_t                   # IDENTICAL syntax, Chapter 22's rule still applies")
print("torch_mean = X_t.mean(dim=0)                # dim= instead of axis=, same MEANING")
print("torch_reshaped = X_t.reshape(2, 6)           # IDENTICAL syntax")
print("X_t.requires_grad_(True)                     # THE key addition: enables autograd tracking")
print("loss = (X_t @ W_t).sum()")
print("loss.backward()                              # automatic Chapter-46-style backprop, no manual chain rule!")
print("print(X_t.grad)                              # the gradient, computed automatically")


### Expected Output (verified on NumPy 2.4.4)
```
NumPy forward pass  : (4, 2)
NumPy mean(axis=0)  : (3,)
NumPy reshape(2, 6) : (2, 6)

--- PyTorch equivalents (same shapes, same axis semantics, near-identical syntax) ---
import torch
X_t = torch.from_numpy(X)                 # zero-copy conversion FROM a numpy array
W_t = torch.from_numpy(W)
torch_result = X_t @ W_t                   # IDENTICAL syntax, Chapter 22's rule still applies
torch_mean = X_t.mean(dim=0)                # dim= instead of axis=, same MEANING
torch_reshaped = X_t.reshape(2, 6)           # IDENTICAL syntax
X_t.requires_grad_(True)                     # THE key addition: enables autograd tracking
loss = (X_t @ W_t).sum()
loss.backward()                              # automatic Chapter-46-style backprop, no manual chain rule!
print(X_t.grad)                              # the gradient, computed automatically
```

### `axis=` vs `dim=` — the one naming difference to internalize
NumPy uses `axis`; PyTorch uses `dim` (TensorFlow uses `axis`, matching NumPy). They mean
**exactly the same thing** — "the dimension this operation collapses/acts along" (Chapter 10).
This is the single most common surface-level naming friction when switching between the
libraries, and now you know it's naming only, not a conceptual difference.

### What Chapter 46's manual backprop actually bought you
Having hand-derived and hand-coded backpropagation for a real 2-layer network (Chapter 46),
gradient-checked against finite differences, means `loss.backward()` in PyTorch is no longer a
magic incantation — it's automating *exactly* the chain-rule bookkeeping you did by hand,
tracking the same `dZ2 = A2 - Y`-style intermediate gradients automatically as operations are
applied, rather than requiring you to derive and write out `dW1`, `db1`, `dW2`, `db2` yourself.
This is the single biggest practical payoff of Chapter 46 existing in this course at all.

### `torch.from_numpy()` — zero-copy, shares memory (Chapter 8's lesson, one more domain)
`torch.from_numpy(X)` creates a tensor that **shares the underlying memory** with the original
NumPy array (when on CPU) — mutating one mutates the other, exactly Chapter 8's view mechanism,
now crossing a library boundary rather than staying within NumPy. This is a genuine, real gotcha
in mixed NumPy/PyTorch pipelines: an in-place NumPy operation on `X` after the conversion can
silently affect `X_t`, and vice versa.

### Interview Insight
> "You're comfortable with NumPy but new to PyTorch. What's actually new to learn, versus what
> transfers directly?"
Shape/broadcasting/axis reasoning, indexing, and most of the array-manipulation API transfer
directly (often with `axis`→`dim` renaming). What's genuinely new: GPU device placement
(`.to('cuda')`), automatic differentiation (`requires_grad`, `.backward()`, replacing manual
backprop like Chapter 46), and the computational-graph mental model underlying autograd — the
data-manipulation fluency built on NumPy is not wasted, it's the majority of the surface area.


---
# CHAPTER 56 — Professional Code Quality

## Concept
"Learning code" proves a concept works. "Production code" additionally validates its inputs,
documents its assumptions, avoids hidden mutation, and fails clearly when misused. This chapter
is a direct before/after applying nearly every lesson from this course to one realistic function.


In [ ]:
# --- LEARNING CODE: works, but has real problems a code review would flag ---
def normalize_learning(X):
    X -= X.mean(axis=0)      # BUG (Chapter 8): mutates the caller's array if X is a view!
    X /= X.std(axis=0)        # BUG (Chapter 36): crashes/produces inf for a constant column
    return X

# --- PRODUCTION CODE: the same operation, hardened using this course's lessons ---
def normalize_production(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Standardize each column of X to zero mean, unit variance.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
        Input data. NOT modified in place.
    eps : float, default 1e-12
        Small constant added to std to avoid division by zero for constant columns.

    Returns
    -------
    np.ndarray, shape (n_samples, n_features)
        A NEW array; X is left unmodified.

    Raises
    ------
    ValueError
        If X is not 2-dimensional, or contains non-finite values.
    """
    X = np.asarray(X)                                          # accept array-likes, Chapter 2
    if X.ndim != 2:                                              # Chapter 3: shape validation
        raise ValueError(f"Expected a 2D array, got shape {X.shape}")
    if not np.isfinite(X).all():                                 # Chapter 15: defensive NaN/inf check
        raise ValueError("Input contains NaN or infinite values")

    X = X.astype(np.float64, copy=True)                          # Chapter 8: EXPLICIT copy, never mutate caller's data
    mean = X.mean(axis=0, keepdims=True)                          # Chapter 10/11: keepdims for correct broadcasting
    std = X.std(axis=0, keepdims=True)
    std_safe = np.where(std < eps, 1.0, std)                       # Chapter 36: guard against constant columns

    return (X - mean) / std_safe


# --- Demonstrating the difference concretely ---
original = np.array([[1., 2., 3.], [3., 2., 3.], [5., 2., 3.]])   # column 1 and 2 are CONSTANT
view_of_original = original[0:2]   # a view, per Chapter 8

print("--- normalize_production (safe) ---")
result = normalize_production(original)
print("Result:\n", result)
print("Original array UNCHANGED:\n", original, " <- proves no in-place mutation")

print("\n--- normalize_learning (unsafe), called on a VIEW ---")
before = original.copy()
normalize_learning(view_of_original)
print("Original array AFTER calling on just a subset view:\n", original)
print("Changed vs before?", not np.array_equal(before, original), " <- the exact Chapter 8 bug, live")


### Expected Output (verified on NumPy 2.4.4)
```
--- normalize_production (safe) ---
Result:
 [[-1.22474487  0.          0.        ]
  [ 0.          0.          0.        ]
  [ 1.22474487  0.          0.        ]]
Original array UNCHANGED:
 [[1. 2. 3.]
  [3. 2. 3.]
  [5. 2. 3.]]  <- proves no in-place mutation

--- normalize_learning (unsafe), called on a VIEW ---
Original array AFTER calling on just a subset view:
 [[-1. nan nan]
 [ 1. nan nan]
 [ 5.  2.  3.]]
Changed vs before? True  <- the exact Chapter 8 bug, live
```

### Reading the production result
Columns 1 and 2 (constant `2` and `3` respectively, across the FULL 3-row `original`) correctly
become all-`0.0` in `normalize_production`'s output, rather than `nan`/`inf` — the `eps` guard
from Chapter 36 doing its job. `original` is provably unchanged after the call — the explicit
`.astype(..., copy=True)` from Chapter 8 doing its job.

### Reading the buggy result — it's worse than "just a wrong value"
`view_of_original` is only the **first two rows** of `original`. Within just those two rows,
columns 1 and 2 (`[2,2]` and `[3,3]`) are *also* constant — so `normalize_learning`'s
unguarded `X /= X.std(axis=0)` divides by exactly zero, and a live `RuntimeWarning: invalid
value encountered in divide` fires. The result isn't merely "wrong," it's `nan` — silently
written back into `original`'s first two rows because of the Chapter 8 mutation bug, while the
untouched third row (`[5, 2, 3]`) remains numerically fine, sitting right next to now-corrupted
`nan` values in the very same array. This is exactly the kind of partially-corrupted,
partially-fine array that's hardest to catch downstream, since a naive `.isnan().any()` check
would catch it here, but a spot-check of "does this look reasonable" on just the last row would
not.

### The checklist this production version actually applies (traced to source chapters)
| Practice | Chapter |
|---|---|
| `np.asarray()` to accept array-likes flexibly | 2 |
| Validate `ndim`/`shape` before proceeding | 3 |
| Check `np.isfinite()` before trusting numeric input | 15 |
| Explicit `copy=True` — never silently mutate caller data | 8 |
| `keepdims=True` for correct broadcast realignment | 10, 11 |
| Guard against division by (near) zero | 36 |
| Docstring stating shapes, mutation behavior, and exceptions | — |
| Type hints signaling expected input/output types | — |

### Interview Insight
> "What's the difference between code that works and code you'd approve in a production code
> review?"
Working code proves the algorithm is correct on a happy-path input. Production code
additionally: validates its inputs rather than assuming them, never silently mutates data the
caller still holds a reference to, handles known edge cases explicitly (constant columns, NaN
inputs) rather than letting them fail unpredictably downstream, and documents its shape/mutation
contract so callers don't have to read the implementation to use it safely.


---
## Chapter 53–56 Summary

- **Performance interview**: the real skill is narrating *why* each optimization step helps
  (loop→comprehension is modest; naive-Python→vectorized-NumPy is the large jump; `out=`-style
  micro-optimization is real but secondary) — and knowing when to stop, not applying every
  possible optimization reflexively.
- **NumPy vs pandas**: a pandas Series/DataFrame column IS a NumPy array underneath (verified:
  `s.mean()` and `np.nanmean(s.values)` agree exactly) — pandas owns heterogeneous/labeled data
  wrangling, NumPy owns the homogeneous numerical core; production pipelines routinely use both
  in sequence.
- **NumPy vs PyTorch/TensorFlow**: shape/broadcasting/axis reasoning transfers directly (mostly
  a `axis`→`dim` renaming for PyTorch); what's genuinely new is GPU placement and autograd
  (`.backward()`), which automates exactly the manual chain-rule work done by hand in Chapter 46
  — `torch.from_numpy()` shares memory with the source array, another instance of Chapter 8's
  view mechanism crossing a library boundary.
- **Professional code quality**: a direct, concrete before/after showing nearly every earlier
  chapter's lesson (input validation, explicit copying, `keepdims`, zero-guards, docstrings)
  converting "code that works" into "code that survives a code review" — with the learning
  version's Chapter 8 mutation bug demonstrated live, not just described.

---

## Mastery Exercises

**Exercise 1:** In the performance progression (v0→v3), which single step produced the largest
speedup, and what specific mechanism (not just "it's vectorized") explains why that step in
particular dwarfs the others?

**Exercise 2:** Name two things you would need to check/change before feeding a pandas
DataFrame directly into a NumPy-based function expecting a homogeneous numerical array.

**Coding Challenge:** Take `normalize_learning` from this chapter and write
`normalize_learning_test_suite()` — a set of assertions (using `np.testing.assert_allclose`
and explicit `try/except` checks) that would have caught both of its bugs (the mutation bug and
the constant-column bug) automatically, without a human needing to notice the corrupted output
by eye.

**Debugging Challenge:** A teammate converts a NumPy array to a PyTorch tensor with
`torch.from_numpy(X)`, later calls `X += 1` (an in-place NumPy operation) on the *original*
array for an unrelated reason, and is confused why their PyTorch tensor's values also changed.
Explain the root cause in one sentence, referencing the exact NumPy chapter this pattern
originates from.

**Interview Questions:**
1. Walk through the optimization progression for a loop-based NumPy performance question, and
   explain what interviewers are really evaluating beyond "did you get the right final speed."
2. What is a pandas Series, structurally, in terms of NumPy concepts already covered?
3. What does `.backward()` in PyTorch automate that you had to do manually in Chapter 46, and
   why does having done it manually first make `.backward()` easier to trust and debug?
4. List four specific differences between "learning code" and "production code" for a NumPy
   function, each tied to a lesson from an earlier chapter.

**Quiz:**
- Q1: True/False — `list comprehension` typically provides a similar order-of-magnitude speedup
  over a `for` loop as true NumPy vectorization does.
- Q2: What NumPy method does `pandas.Series.mean()` conceptually mirror for NaN-handling?
- Q3: In PyTorch, what keyword argument name is used where NumPy uses `axis`?

---
